In [14]:
from google import genai
import chromadb
from dotenv import load_dotenv
import os

In [15]:
load_dotenv()

# Initialize the GenAI client
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

Manual chunking

In [16]:
from pypdf import PdfReader

def extract_text(file_path):
    # loading pdf
    reader = PdfReader(file_path)
    full_text = ""

    # retreive page texts
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"
    
    return full_text

def chunk_text(full_text,  chunk_size=500, chunk_overlap=50):
    chunks = []

    for i  in range(0, len(full_text) , chunk_size - chunk_overlap):
        chunk = full_text[i: i + chunk_size]
        chunks.append(chunk)
    return chunks

In [17]:
full_text = extract_text(file_path="karthi_profile_summary.pdf")
chunks = chunk_text(full_text=full_text)

Embedding

In [18]:
def embed_chunks(chunks):
    embeddings = []
    for chunk in chunks:
        result = client.models.embed_content(
            model="gemini-embedding-2",
            contents=chunk
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings


In [19]:
chroma_client = chromadb.PersistentClient(path="./rag_db")
collection = chroma_client.get_or_create_collection(name = "knowledge_base")
embeddings = embed_chunks(chunks=chunks)
print(embeddings)

chunk_ids = [ f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    embeddings=embeddings,
    ids=chunk_ids,
    documents=chunks
)



[[-0.009491359, 0.021164007, -0.018607674, 0.02570754, 0.014747469, -0.00073750457, -0.007026655, -0.0044971877, 0.025999337, -0.049902335, -0.017307447, 0.0056512817, 0.005604477, -0.021924824, -0.0012296097, -0.018285621, 0.036975034, -0.017595455, 0.00990301, -0.006109054, -0.015488853, -0.033459917, -0.02597227, 0.0026210153, -0.021736491, 0.016934693, -0.0040076515, -0.0043362165, -0.023782467, 0.138028, -0.025349356, -0.020111373, 0.0035342262, -0.010676692, 0.019580912, -0.018807195, 0.008509963, 0.0011138876, -0.029423514, 0.013078828, 0.0080083, 0.012136022, -0.0014668307, 0.0017319074, 0.006120898, 0.0014590423, -0.018356208, -0.0043791556, 0.011792313, -0.009116838, -0.0064337268, -0.017301317, 0.047416374, 0.0065670516, -0.0016200661, 0.015053869, 0.03078821, -0.007958101, 0.007996852, 0.0040778043, 0.031730805, -0.0044532097, 0.022576878, -0.010350888, -0.0014713504, -0.016818019, 0.001850833, -0.0021895352, 0.00027900387, 0.021366712, 0.014349027, 0.002189257, -0.02050235

Retrival pipeline

In [20]:
query = "tell me about karthi's skills"

#embedding the user query
query_embeddings = client.models.embed_content(
    model="gemini-embedding-2",
    contents=query
).embeddings[0].values

#querying and retrieving relevant data from the ChromaDB
query_result = collection.query(
    query_embeddings=query_embeddings,
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(query_result)


{'ids': [['chunk_0', 'chunk_1', 'chunk_5']], 'embeddings': None, 'documents': [['Karthi D — Developer Profile\nThis profile was generated using the available conversation context and known technical interests, learning paths, and\nprofessional experience shared previously.\nProfessional Summary\nBackend-focused software developer with experience in Python and Django development, actively expanding\nexpertise into AWS, DevOps, AI/ML, Angular, MongoDB, and scalable backend systems. Experienced working in\nproduction environments involving Django and Tornado architectures, API develo', 'lving Django and Tornado architectures, API development, authentication systems, RAG\npipelines, and deployment workflows.\nCore Skills & Interests\n\x7f\nPython Development\n\x7f\nDjango Backend Development\n\x7f\nJWT Authentication\n\x7f\nREST API Development\n\x7f\nMongoDB\n\x7f\nAWS Fundamentals\n\x7f\nDevOps Learning Path\n\x7f\nAI & Machine Learning Fundamentals\n\x7f\nRAG Systems with ChromaDB\n\x7f

In [21]:
# 1. Safely extract and filter out any None values
retrieved_chunks = [c for c in query_result.get("documents", [[]])[0] if c is not None]

# 2. Check if you actually have data before calling the model
if not retrieved_chunks:
    print("Warning: No documents were retrieved. Check your ChromaDB storage.")
    context_text = "No relevant context found."
else:
    context_text = "\n".join(retrieved_chunks)

# 3. Use the correct prompt structure for google-genai
response = client.models.generate_content(
    model="gemini-2.5-flash", # Use a currently available model like 2.0-flash
    contents=f"Context: {context_text}\n\nQuestion: tell me about karthi's experience"
)
print(response.text)

Based on the provided profile, here's a summary of Karthi's experience:

Karthi is a **backend-focused software developer** with practical experience primarily in **Python and Django development**.

His experience includes:
*   Working in **production environments** using **Django and Tornado architectures**.
*   **API development** (REST APIs, FastAPI).
*   Implementing **authentication systems** (JWT Authentication).
*   Developing **RAG pipelines** (specifically with ChromaDB).
*   Working with **MongoDB**.
*   Handling **deployment workflows** and understanding **infrastructure concepts**.

He is currently a **working professional** and is actively expanding his expertise into areas like AWS, DevOps, AI/ML, and learning Angular.
